# 🛒 Retail Dataset Analysis 02 - Customer, Pricing & Retention Analysis

Following a previous analysis on performence and product mix, this notebook performs a deep-dive exploratory analysis into retail sales, focusing on the relationship between customer and pricing data, as well as validating pre-entered kpi data such as predicted_CLV and loyalty_score.

#### Key Objectives:

* **Category Mix Evolution: EDIT** 

---

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("RetailEDA") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

df = spark.read.csv(
    "../synthetic-datasets/datasets/retail/synthetic_retail_20250901.csv",
    header=True,
    inferSchema=True
)

df.printSchema()
df.show(5)

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- store_location: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_value: double (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- customer_income: double (nullable = true)
 |-- loyalty_score: double (nullable = true)
 |-- days_since_last_purchase: integer (nullable = true)
 |-- purchase_frequency_monthly: double (nullable = true)
 |-- average_order_value: double (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- sales_channel: string (nullable = true)
 |-- season: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- customer_registration_date: date (nullable = true)
 |-- customer_segment: integer (nullable = true)
 |-- pre

### 📊 Strategic Customer Behavior & Retention Analysis

1. **CLV Validation:** Made a validation analysis of the predicted_clv field to see if it can be used for future modeling.
    - 1.1. **customer_segment:** It was found this field has no predictive or discriminatory power for revenue behavior since the top spenders are scattared across all segments in almost exact proportions and should be re-evaluated in Gold layer
    - 1.2. **predicted_clv & loyalty_score:** It was found these fields have almost no relationship with how the customers actually behave -- almost zero correlation with total_value, almost zero correlation with purchase_frequency_monthly, barely any correlation with avg_order_value -- should be re-evaluated in Gold layer
2. **Tenure Analysis:** Identify patterns in transaction_count and average_spending by customer loyalty age.
    - 2.1. **first_registration_date:** In the previous EDA, it was found that all customer_ids have multiple registration_dates, so a canonical one was created as baseline for tenure
    - 2.2. **Bell Curve** It was found through Tenure that the average basket remains consistent regardless of time, indicating few brand new customers, few original/ old customers and a lot of middle-aged customers sustaining the business
    - 2.3. **Flat Line:** This analysis also says that I cannot rely on customer value to grow my CLV, since avg_basket doesn't grow and retention starts to slowly fall off near the last transaction_dates in the dataset. Other analysis are necessary to understand what does make frequency increase and recover retention over time
    - 2.4. **Churn Rates:** Shows that 56% of the customers are on a stable retention rate, but at least 35% are on a risk level to abandon the shop, while almost 10% already churned over
3. **Sales Channel & Store Cross-Analysis** Identify which possible sales_channel and store_location leads to higher spending
    - 3.1. **Income:** It was found that the average_order_value still remains the same despite whatsoever combination of sales_channel & store_location, but it also was found that clients tend to buy "In-store" much more than other channels, with a clear bias toward large, physical marketplaces like Shopping Malls rather than smaller venues like Outlets
    - 3.2. **Systemic Risk:** Further analysis into the previous health custom field tells us that the churn is systemic; every store_location is experiencing the same decline in retention
4. **Discount Intelligence** Identify the efficacy of discounts being applied over time.
    - 4.1. **Discount Strategy:** Initial analysis reveals that every single transaction includes a discount, though the average depth is under 1%. That suggests that discount functions as a strategic lever for handling taxes, fixed fees or rounding, rather than an actual promotional tool

In [3]:
#   ═════════════════════════════════════════════════════════════════════════════
#   CLV Validation
#   Is predicted_clv consistent with observed transaction behavior?
#   ═════════════════════════════════════════════════════════════════════════════

# Check if high-value segments (e.g. = 0) are actually spending more

customer_segment_analysis = df.groupBy("customer_segment").agg(
    F.avg("total_value").alias("avg_transaction_value"),
    F.sum("total_value").alias("total_revenue_contribution"),
    F.avg("purchase_frequency_monthly").alias("avg_frequency"),
    F.count("*").alias("segment_size")
).orderBy(F.desc("total_revenue_contribution"))

customer_segment_analysis.show()

+----------------+---------------------+--------------------------+------------------+------------+
|customer_segment|avg_transaction_value|total_revenue_contribution|     avg_frequency|segment_size|
+----------------+---------------------+--------------------------+------------------+------------+
|               0|    66.35505774605788|       3.030170067031479E7|15.101690951808425|      456660|
|               1|     66.4358541427228|      2.0681414958775464E7|15.519964527644811|      311299|
|               2|    64.52573981244542|        2043981.8600388337|15.644557606619987|       31677|
|               3|    68.33038525216665|         24872.26023178866| 18.96694214876033|         364|
+----------------+---------------------+--------------------------+------------------+------------+



In [ ]:
top_threshold = df.select(F.percentile_approx("total_value", 0.99)).collect()[0][0]
print(f"Top 1% Spend Threshold: {top_threshold}")

df.filter(F.col("total_value") > top_threshold) \
  .groupBy("customer_segment") \
  .count() \
  .orderBy(F.desc("count")) \
  .show()

Top 1% Spend Threshold: 559.4930696431886
+----------------+-----+
|customer_segment|count|
+----------------+-----+
|               0| 4606|
|               1| 3204|
|               2|  266|
|               3|    4|
+----------------+-----+



In [ ]:
df.select(
    corr("predicted_clv", "total_value").alias("clv_vs_total_value"),
    corr("predicted_clv", "purchase_frequency_monthly").alias("clv_vs_frequency"),
    corr("predicted_clv", "average_order_value").alias("clv_vs_avg_order_value")
).show()

df.select(
    corr("loyalty_score", "total_value").alias("loyalty_vs_total_value"),
    corr("loyalty_score", "purchase_frequency_monthly").alias("loyalty_vs_frequency"),
    corr("loyalty_score", "average_order_value").alias("loyalty_vs_avg_order_value")
).show()

+--------------------+--------------------+----------------------+
|  clv_vs_total_value|    clv_vs_frequency|clv_vs_avg_order_value|
+--------------------+--------------------+----------------------+
|-0.00195058747939...|0.001332813717515...|  0.024040213128206752|
+--------------------+--------------------+----------------------+

+----------------------+--------------------+--------------------------+
|loyalty_vs_total_value|loyalty_vs_frequency|loyalty_vs_avg_order_value|
+----------------------+--------------------+--------------------------+
|  -0.00202657938427...|-0.09761098851106238|      4.419299396125132E-4|
+----------------------+--------------------+--------------------------+



In [ ]:
#   ═════════════════════════════════════════════════════════════════════════════
#   Customer Tenure Analysis
#   ═════════════════════════════════════════════════════════════════════════════

# It was found in the first EDA that all customers have multiple registration dates,
# So I'll make a simple transformation by creating a canonical customer table (without touching the transactions) as reference for my analysis

canonical_df = df \
    .groupBy("customer_id") \
    .agg(F.min("customer_registration_date").alias("first_registration_date"))

print(f"Canonical customer count: {canonical_df.count()}")
print(f"Unique customers in raw: {df.select('customer_id').distinct().count()}")
print(f"Total transactions (raw): {df.count()}")

df_tenure = df \
    .drop("customer_registration_date") \
    .join(canonical_df, on="customer_id", how="left")

print(f"Total transactions (after join): {df_tenure.count()}")

Canonical customer count: 40000
Unique customers in raw: 40000
Total transactions (raw): 800000
Total transactions (after join): 800000


In [16]:
df_tenure_final = df_tenure.withColumn("tenure_month_start", 
    (F.floor(F.datediff(F.col("transaction_date"), F.col("first_registration_date")) / 30))
)

df_tenure_final = df_tenure_final.withColumn("tenure_bucket", 
    F.concat(F.col("tenure_month_start").cast("string"), F.lit("-"), 
             (F.col("tenure_month_start") + 1).cast("string"), F.lit(" Months"))
)

tenure_analysis = df_tenure_final.groupBy("tenure_month_start", "tenure_bucket").agg(
    F.count("*").alias("tx_count"),
    F.avg("total_value").alias("avg_basket"),
    F.sum("total_value").alias("total_rev")
).orderBy("tenure_month_start")

tenure_analysis.select("tenure_bucket", "tx_count", "avg_basket", "total_rev").show(80, truncate=False)

+-------------+--------+------------------+------------------+
|tenure_bucket|tx_count|avg_basket        |total_rev         |
+-------------+--------+------------------+------------------+
|5-6 Months   |1       |21.623190054459304|21.623190054459304|
|8-9 Months   |1       |12.07038570642153 |12.07038570642153 |
|9-10 Months  |3       |258.41893333179604|775.2567999953881 |
|10-11 Months |3       |45.35928915900147 |136.07786747700442|
|11-12 Months |4       |56.90655856748873 |227.62623426995492|
|12-13 Months |19      |52.33910961856619 |994.4430827527576 |
|13-14 Months |29      |45.33860484727466 |1314.819540570965 |
|14-15 Months |53      |65.14021773001987 |3452.431539691053 |
|15-16 Months |96      |52.18468162940869 |5009.729436423235 |
|16-17 Months |162     |70.74584028693565 |11460.826126483575|
|17-18 Months |266     |58.00608304887148 |15429.618090999815|
|18-19 Months |422     |65.09589547200021 |27470.46788918409 |
|19-20 Months |741     |67.56468467217017 |50065.431342

In [41]:
snapshot_date = df_tenure_final.select(F.max("transaction_date")).collect()[0][0]

recency_df = df_tenure_final.groupBy("customer_id") \
    .agg(F.max("transaction_date").alias("last_purchase_date")) \
    .withColumn("days_since_last", F.datediff(F.lit(snapshot_date), F.col("last_purchase_date")))

recency_df = recency_df.withColumn("health_status", 
    F.when(F.col("days_since_last") <= 30, "1. Active (0-30d)")
     .when(F.col("days_since_last") <= 60, "2. Low Risk (31-60d)")
     .when(F.col("days_since_last") <= 90, "3. High Risk (61-90d)")
     .when(F.col("days_since_last") <= 180, "4. Churning (91-180d)")
     .otherwise("5. Churned (>180d)")
)

churn_summary = recency_df.groupBy("health_status").agg(
    F.count("customer_id").alias("customer_count"),
    F.avg("days_since_last").alias("avg_days_inactive")
).withColumn(
    "percent_total",
    F.round((F.col("customer_count") / F.sum("customer_count").over(Window.partitionBy()) * 100), 2)
).select("health_status", "customer_count", "percent_total", "avg_days_inactive").orderBy("health_status")
  
churn_summary.show(truncate=False)

+---------------------+--------------+-------------+------------------+
|health_status        |customer_count|percent_total|avg_days_inactive |
+---------------------+--------------+-------------+------------------+
|1. Active (0-30d)    |22704         |56.76        |12.875264270613108|
|2. Low Risk (31-60d) |9727          |24.32        |43.437647784517324|
|3. High Risk (61-90d)|4266          |10.67        |73.4423347398031  |
|4. Churning (91-180d)|3005          |7.51         |118.01996672212978|
|5. Churned (>180d)   |298           |0.75         |215.20469798657717|
+---------------------+--------------+-------------+------------------+



In [55]:
#   ═════════════════════════════════════════════════════════════════════════════
#   Sales Channel & Store Cross-Analysis
#   ═════════════════════════════════════════════════════════════════════════════

# Check if the $66 average_order_value is consistent across channels and stores, or if it's hiding some high-performing pockets of the business

channel_store_analysis = df_tenure_final.groupBy("sales_channel", "store_location").agg(
    F.count("*").alias("tx_count"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.avg("total_value").alias("avg_basket"),
    F.sum("total_value").alias("total_revenue")
).withColumn(
    "tx_per_customer", F.round(F.col("tx_count") / F.col("unique_customers"), 2)
).orderBy(F.desc("tx_count"))

channel_store_analysis.show(40, truncate=False)


+-------------+--------------+--------+----------------+-----------------+------------------+---------------+
|sales_channel|store_location|tx_count|unique_customers|avg_basket       |total_revenue     |tx_per_customer|
+-------------+--------------+--------+----------------+-----------------+------------------+---------------+
|In-Store     |Shopping Mall |143394  |38886           |65.98742481286037|9462200.7936153   |3.69           |
|In-Store     |Downtown      |119753  |38025           |66.23465802214676|7931799.002126141 |3.15           |
|In-Store     |Strip Mall    |96082   |36276           |66.58728157467482|6397839.188257906 |2.65           |
|In-Store     |Standalone    |72213   |33465           |66.26955995450223|4785523.73299447  |2.16           |
|Online       |Shopping Mall |59977   |31117           |67.1916626879419 |4029954.353034691 |1.93           |
|Online       |Downtown      |49530   |28396           |66.18753300976918|3278268.509973868 |1.74           |
|In-Store 

In [47]:
store_tx_analysis = df_tenure_final.groupBy("store_location").agg(
    F.count("*").alias("tx_count"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.avg("total_value").alias("avg_basket"),
    F.sum("total_value").alias("total_revenue")
).withColumn(
    "tx_per_customer", F.round(F.col("tx_count") / F.col("unique_customers"), 2)
).orderBy(F.desc("tx_count"))

store_tx_analysis.show(40, truncate=False)

+--------------+--------+----------------+-----------------+--------------------+---------------+
|store_location|tx_count|unique_customers|avg_basket       |total_revenue       |tx_per_customer|
+--------------+--------+----------------+-----------------+--------------------+---------------+
|Shopping Mall |239302  |39896           |66.38853498379642|1.5886909198692452E7|6.0            |
|Downtown      |199324  |39733           |66.2977072624948 |1.3214724202389512E7|5.02           |
|Strip Mall    |160245  |39257           |66.30716154865965|1.0625391102364965E7|4.08           |
|Standalone    |119928  |38020           |66.31610773612574|7953158.168578088   |3.15           |
|Outlet        |80001   |34512           |66.16005257135119|5292870.365760666   |2.32           |
|NULL          |1200    |1179            |65.76392631262476|78916.71157514972   |1.02           |
+--------------+--------+----------------+-----------------+--------------------+---------------+



In [48]:
channel_tx_analysis = df_tenure_final.groupBy("sales_channel").agg(
    F.count("*").alias("tx_count"),
    F.countDistinct("customer_id").alias("unique_customers"),
    F.avg("total_value").alias("avg_basket"),
    F.sum("total_value").alias("total_revenue")
).withColumn(
    "tx_per_customer", F.round(F.col("tx_count") / F.col("unique_customers"), 2)
).orderBy(F.desc("tx_count"))

channel_tx_analysis.show(40, truncate=False)

+-------------+--------+----------------+-----------------+--------------------+---------------+
|sales_channel|tx_count|unique_customers|avg_basket       |total_revenue       |tx_per_customer|
+-------------+--------+----------------+-----------------+--------------------+---------------+
|In-Store     |479223  |40000           |66.24776647192618|3.174745339197588E7 |11.98          |
|Online       |200667  |39754           |66.49104298739655|1.3342558123151904E7|5.05           |
|Mobile App   |80121   |34620           |66.49757054300981|5327851.849476489   |2.31           |
|Phone        |24090   |17999           |65.09265404945104|1568082.0360512757  |1.34           |
|Catalog      |15899   |13077           |67.04977348923215|1066024.348705302   |1.22           |
+-------------+--------+----------------+-----------------+--------------------+---------------+



In [59]:
df_health = df_tenure_final.join(
    recency_df.select("customer_id", "health_status"),
    on="customer_id",
    how="left"
)

at_risk_locations = df_health.filter(F.col("health_status").contains("Risk")).groupBy("store_location").agg(
    F.countDistinct("customer_id").alias("at_risk_customers"),
    F.sum("total_value").alias("at_risk_revenue_exposure")
).orderBy(F.desc("at_risk_customers"))

at_risk_locations.show(truncate=False)


+--------------+-----------------+------------------------+
|store_location|at_risk_customers|at_risk_revenue_exposure|
+--------------+-----------------+------------------------+
|Shopping Mall |13953            |5375866.435839705       |
|Downtown      |13891            |4522933.703525778       |
|Strip Mall    |13725            |3591570.513489083       |
|Standalone    |13275            |2708055.252034487       |
|Outlet        |11931            |1786003.2036767558      |
|NULL          |410              |29825.54770041118       |
+--------------+-----------------+------------------------+



In [ ]:
#   ═════════════════════════════════════════════════════════════════════════════
#   Pricing Intelligence
#   ═════════════════════════════════════════════════════════════════════════════

# Apply discount intelligence analysis to identify if I'm buying my customers' loyalty with discounts

df_discounts = df_health.withColumn(
    "is_discounted", F.when(F.col("discount_percentage") > 0, 1).otherwise(0)
).withColumn(
    "discount_rate", F.round(F.col("discount_percentage") / (F.col("total_value") + F.col("discount_percentage")), 4)
)

discount_analysis = df_discounts.groupBy("health_status").agg(
    F.count("*").alias("total_tx"),
    F.round(F.avg("is_discounted") * 100, 2).alias("perc_tx_discounted"),
    F.round(F.avg("discount_rate") * 100, 2).alias("avg_discount_depth"),
    F.avg("total_value").alias("avg_basket")
).orderBy("health_status")

discount_analysis.show(truncate=False)

+---------------------+--------+------------------+------------------+-----------------+
|health_status        |total_tx|perc_tx_discounted|avg_discount_depth|avg_basket       |
+---------------------+--------+------------------+------------------+-----------------+
|1. Active (0-30d)    |469350  |100.0             |0.97              |66.54128486153193|
|2. Low Risk (31-60d) |192302  |100.0             |0.97              |66.33759425788048|
|3. High Risk (61-90d)|80560   |100.0             |0.97              |65.26070761280138|
|4. Churning (91-180d)|53339   |100.0             |0.97              |65.76938811443634|
|5. Churned (>180d)   |4449    |100.0             |0.96              |67.09140271942076|
+---------------------+--------+------------------+------------------+-----------------+



### ═══════════════════════════════════════════════════════════════
#### DATA QUALITY LOG — Silver Layer Transformation Spec
#### Generated from: 01_eda_product_mix_and_growth.ipynb + 02_eda_customer_behavior_and_retention.ipynb
### ═══════════════════════════════════════════════════════════════

- DQ-001: customer_income — NaN values present → impute with median per customer_segment
- DQ-002: season column — mismatches derived calendar season → replace with derived
- DQ-003: total_value integrity — validate equals: unit_price * quantity * (1 - discount_pct)
- DQ-004 [CRITICAL]: customer_registration_date — multiple values per customer_id → canonical first_registration_date derived via groupBy+min, all transactions preserved. Silver must persist canonical date only, drop raw column
- DQ-005: customer_segment / predicted_clv / loyalty_score — near-zero correlation with actual behavior → flag as unreliable, do NOT use in Gold aggregations as-is, re-derive in Gold layer from transaction history